In [2]:
from dotenv import load_dotenv

load_dotenv()

import os

In [ ]:
prompt_template = """
Bạn là chuyên gia chuyển đổi truy vấn từ Power BI sang HIVE SQL.

tôi có file excel, gồm 2 sheet. "Key" sheet là bảng ánh xạ từ bảng trong power Bi  DAX sang bảng và cột của HIVE table. "Measure" sheet là danh sách các công thức đo được tạo trên power BI. Bạn là DA engineer, hãy chuyển đổi cột DAX sang SQL query trên Trino - HIVE . Tự ánh xạ bảng dựa vào "Key" sheet. Tạo table gồm các cột "measure_index", "Measure"."table", "Measure"."name", "sql_hive" tương ứng với cột "dax".
  Trong sheet "Key" , 
Table_BPI là tên bảng trong power BI, 
Table_HIVE là tên bảng trong Trino Hive
Field_BPI là tên cột trong power BI
Field_HIVE là tên cột trong Trino Hive
Tạo tất cả các SQL query tương ứng với cột "Measure"."Dax"

Bảng Ánh xạ bảng bên Power BI(Table_PBI) sang bảng Hive SQL (Table_HIVE), và cột Power BI (Field_PBI) sang cột HIVE SQL (Field_HIVE)
| Table_BPI        | Table_HIVE                               | Field_PBI                               | Field_HIVE      |
|------------------|------------------------------------------|------------------------------------------|-----------------|
| Chi phí          | hive.bitu_silver_data.actual_cost         | Trung tâm CP                             | department      |
| Chi phí          | hive.bitu_silver_data.actual_cost         | Thị trường                               | territory       |
| Chi phí          | hive.bitu_silver_data.actual_cost         | Tháng                                    | month           |
| Chi phí          | hive.bitu_silver_data.actual_cost         | SPDV                                     | old_category    |
| Chi phí          | hive.bitu_silver_data.actual_cost         | Ngày                                     | report_date     |
| Chi phí          | hive.bitu_silver_data.actual_cost         | Mã SPDV                                  | category_code   |
| Chi phí          | hive.bitu_silver_data.actual_cost         | Loại Chi phí                             | cost_group      |
| Chi phí          | hive.bitu_silver_data.actual_cost         | chi phí                                  | actual_cost     |
| kế hoạch chi phí | hive.bitu_silver_data.plan_cost           | Ngày                                     | plan_date       |
| kế hoạch chi phí | hive.bitu_silver_data.plan_cost           | Tháng                                    | month           |
| kế hoạch chi phí | hive.bitu_silver_data.plan_cost           | Mã KM phí                                | cost_code       |
| kế hoạch chi phí | hive.bitu_silver_data.plan_cost           | Khoản mục cấp 2                          | cost_group      |
| kế hoạch chi phí | hive.bitu_silver_data.plan_cost           | Triệu đồng                               | expected_cost   |
| doanh thu        | hive.bitu_silver_data.kinhdoanh           |                                          | license         |
| doanh thu        | hive.bitu_silver_data.kinhdoanh           | doanh thu cuối                           | revenue         |
| doanh thu        | hive.bitu_silver_data.kinhdoanh           | Khách hàng                               | company_name    |
| doanh thu        | hive.bitu_silver_data.kinhdoanh           | Mã SPDV                                  | category_code   |
| doanh thu        | hive.bitu_silver_data.kinhdoanh           | Ngày xuất hóa đơn                        | report_date     |
| doanh thu        | hive.bitu_silver_data.kinhdoanh           | Khách hàng                               | customer        |
| doanh thu        | hive.bitu_silver_data.kinhdoanh           | Nhóm doanh thu                           | segment         |
| doanh thu        | hive.bitu_silver_data.kinhdoanh           | DT dòng tiền đều/ DT lên 1 lần           | producttype    |
| doanh thu        | hive.bitu_silver_data.kinhdoanh           | Phân loại KH                             | channel         |
| doanh thu        | hive.bitu_silver_data.kinhdoanh           | Phân loại SOC (SOC và non-SOC)           | is_soc          |
| doanh thu        | hive.bitu_silver_data.kinhdoanh           | Phân loại SP/DV                          | group_spdv      |
| doanh thu        | hive.bitu_silver_data.kinhdoanh           | Phòng                                    | department      |
| doanh thu        | hive.bitu_silver_data.kinhdoanh           | SPDV cụ thể                              | old_category    |
| doanh thu        | hive.bitu_silver_data.kinhdoanh           |                                          | category        |
| doanh thu        | hive.bitu_silver_data.kinhdoanh           | Nhóm khách hàng                          | segment3       |
| doanh thu        | hive.bitu_silver_data.kinhdoanh           | Tháng                                    | month           |
| doanh thu        | hive.bitu_silver_data.kinhdoanh           | VAT                                      | vat             |
| doanh thu        | hive.bitu_silver_data.kinhdoanh           | Xuất hóa đơn (HD)/ tạm tính (TT)         | payment_stage  |
| doanh thu        | hive.bitu_silver_data.kinhdoanh           | AM                                       | am              |
| doanh thu        | hive.bitu_silver_data.kinhdoanh           | Presale                                  | presale        |

Câu lệnh tạo measure trên Power BI:
#Power_BI#

Dữ liệu mong muốn đầu ra là chỉ câu lệnh HIVE SQL tương ứng.

"""

In [13]:
os.getenv("VCS_LLM_URL")

'https://secagi.viettelcyber.com/models/gpt-oss-120b-it'

In [14]:
from openai import OpenAI
import json

client = OpenAI(
    api_key=os.getenv("VCS_LLM_API_KEY"),
    base_url=os.getenv("VCS_LLM_URL")
)

In [15]:


def generate_markdown_report( user_query: str) -> str:
    prompt = prompt_template.replace("#Power_BI#", user_query)
    response = client.chat.completions.create(
            model=None,
            messages=[
                {"role": "system", "content": "Bạn là chuyên gia BI & Data Analyst."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.2
        )

    return response.choices[0].message.content.strip()


In [16]:
user_query = """
var MaxDateToToday = CALCULATE(
                        MAX('doanh thu công ty'[Ngày]), 
                        FILTER(
                            ALLSELECTED('Date'[Date]),
                            'Date'[Date] <= MAX('Kế hoạch chi phí'[Ngày]))
                    )
var dateReport =  EOMONTH(MaxDateToToday, 0)
var cpnam  =  CALCULATE(
    SUM('Kế hoạch chi phí'[Triệu đồng]),
    FILTER(
        'Date',
        'Date'[Date] <= MAX('Kế hoạch chi phí'[Ngày]) &&
        YEAR('Date'[Date]) = YEAR(dateReport) && MONTH('Date'[Date]) <= MONTH(dateReport)
    )
)
return cpnam
"""

markdown_output = generate_markdown_report(user_query)
print(markdown_output)

AuthenticationError: Error code: 401 - {'error': {'message': "Authentication Error, LiteLLM Virtual Key expected. Received=rnd_vcs_120b_1111, expected to start with 'sk-'.", 'type': 'auth_error', 'param': 'None', 'code': '401'}}

In [ ]:
import pandas as pd